# VietMedBridge — 05: Source → nhãn đã duyệt → hard negatives → QLoRA → dev F2

Giai đoạn supervised trong master §§13/27/39/56. **04 chạy pretrained và
submit được ngay**, không cần chạy 05 trước khi có nhãn. Chọn runtime GPU
mới; L4/A100 thuận tiện hơn cho training. Qwen8B nạp NF4, batch=1,
gradient checkpointing/accumulation.

Chưa có nhãn: notebook tự tách các nhóm source/đoạn trùng trước, chọn tối
đa 256 source samples train, 40 dev và 40 held-out. Qwen 4B tạo câu hỏi VI
**chỉ cho train**, có exact evidence quote, checkpoint từng mẫu.
Dev/held-out để trống câu hỏi cho team viết độc lập trên các nhóm source
đã reserve. Không thêm teacher mới. Model outputs luôn là draft.

Notebook tải source_review.json để team duyệt: chỉ sửa review fields,
ACCEPT/REJECT + reviewer + query + exact evidence_quote. Dev/held-out yêu cầu
independently_written=true sau khi người review tự viết câu hỏi.
Lưu file đã duyệt về đúng review_path trên Drive và Run all lại.
Human review: tối thiểu 32 train/8 dev/8 held-out được accept, hết PENDING mới xuất labels.
Không tự bật reviewed hoặc exhaustive_chunks cho model predictions.

Nếu Sếp giao Codex duyệt bộ pilot: file có review_mode=AI_ASSISTED_PILOT,
reviewer_type=AI và nguồn gốc ủy quyền được ghi rõ. Bộ pilot vẫn cần đủ
32 train/8 dev/8 held-out; các mẫu PENDING còn lại được để dành, không
biến thành nhãn âm tính. Dev/held-out ghi AI_REVIEWED_PILOT, không giả
independently_written=true hoặc nhãn human gold. Upload file
source_review_ai_pilot.json vào /content bằng tab Files của Colab rồi
Run all. Cell chuẩn bị tự giữ backup và đưa bản duyệt vào đúng Drive path.

Train/dev tách theo query, IDs và nội dung không trùng 1.200 contest queries.
Mine 1 positive + 7 negatives, ưu tiên top ranks, không lấy predictions làm gold.
QLoRA checkpoint giữ optimizer/RNG để resume. Đánh giá dev NDCG/MRR rồi
chạy finalists qua full pipeline và CPU F2 cutoff sweep. Chọn adapter theo
dev F2. Tự chạy dev ablations, rồi chấm held-out với adapter/cutoff đã
freeze; không chọn model bằng held-out score và không promote corpus.
04 tự nhận selected_adapter/calibrated_policy trong namespace mới.

V6: corpus embeddings được tìm và kiểm hash giữa 04/05 và các experiment;
không tính lại corpus chỉ vì đổi tập query. Mining cũ được kiểm và replay
trước khi nạp GPU. Mining mới dùng retrieval rộng, không chạy cascade 8B
để tạo file chờ nhãn. Qwen reranker 8B vẫn dùng cho QLoRA/dev/held-out/04.
Batch inference theo VRAM, có OOM backoff. Final adapter trùng checkpoint
cuối được đánh giá một lần. Tám research ablations được để sau kết quả
đầu tiên; API run_training_workflow(..., run_ablations=True) vẫn hỗ trợ.
Thời gian từng bước ghi training/<run>/runtime_profile.json; từng embedding
pass ghi runtime_profile.json bên cạnh parts. Các source/weights/hash cũ giữ nguyên.


## 1. Bootstrap — CPU


In [ ]:
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError(
        "Dùng Python 3.11–3.13. Trong Runtime > Change runtime type, "
        "chọn phiên bản runtime có Python trong khoảng này."
    )
if importlib.util.find_spec("google.colab") is None:
    raise RuntimeError("Notebook này được thiết kế cho Google Colab.")

from google.colab import drive
# Chạy lại cell này sau MỖI runtime mới/restart: package chỉ được cài trong phiên Colab.
drive.mount("/content/drive")

# Giữ DATA_ROOT đã dùng ở notebook 00–03.
DATA_ROOT = Path("/content/drive/MyDrive/VietMedBridge/data")
WORK_DIR = Path("/content/vmb_work")  # temp files/spill ở ổ local của Colab
DATA_ROOT.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

REPO_URL = "https://github.com/Platypus27-coder/VietMedBridge.git"
CODE_REVISION = None  # Có thể đặt full commit SHA; mặc định dùng code_lock đã lưu.
EXPECTED_PIPELINE_API = 2
CHECKOUT = Path("/content/VietMedBridge")
lock_path = DATA_ROOT / "training_code_lock.json"
lock = json.loads(lock_path.read_text()) if lock_path.exists() else {}
if lock and lock.get("repo_url") != REPO_URL:
    raise ValueError("DATA_ROOT đang khóa vào một repo khác.")
if lock and lock.get("pipeline_api") != EXPECTED_PIPELINE_API and not CODE_REVISION:
    raise RuntimeError(
        "DATA_ROOT đang dùng code cũ. Để nâng lên data-v2, đặt CODE_REVISION='main' "
        "ở cell này rồi chạy lại; dùng tên crawl/build run mới. Raw cũ vẫn được giữ."
    )
RETRIEVAL_WORKFLOW_API = "full-master-plan-supervised-v6-shared-cache-review"
upgrade = lock.get("workflow_api") != RETRIEVAL_WORKFLOW_API
reference = CODE_REVISION or ("main" if upgrade else lock.get("git_commit")) or "main"
if not (CHECKOUT / ".git").exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(CHECKOUT)], check=True)
remote = subprocess.check_output(
    ["git", "-C", str(CHECKOUT), "remote", "get-url", "origin"], text=True
).strip()
if remote != REPO_URL:
    raise ValueError("Checkout hiện tại không thuộc repo VietMedBridge.")
subprocess.run(["git", "-C", str(CHECKOUT), "fetch", "--depth", "1", "origin", reference], check=True)
CODE_COMMIT = subprocess.check_output(
    ["git", "-C", str(CHECKOUT), "rev-parse", "FETCH_HEAD"], text=True
).strip()
if "vietmedbridge" in sys.modules and globals().get("_VMB_IMPORTED_COMMIT") != CODE_COMMIT:
    raise RuntimeError("Mã nguồn đổi trong runtime đã import package. Restart session rồi chạy lại notebook.")
subprocess.run(["git", "-C", str(CHECKOUT), "checkout", "--detach", "FETCH_HEAD"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-e", ".[notebook,retrieval,strong]"],
    cwd=CHECKOUT,
    check=True,
)
PACKAGE_SOURCE = CHECKOUT / "src"
if not (PACKAGE_SOURCE / "vietmedbridge" / "__init__.py").is_file():
    raise RuntimeError(
        f"Repo checkout thiếu package source: {PACKAGE_SOURCE}; commit={CODE_COMMIT}. "
        "Kiểm tra lại git fetch/checkout ở output của cell Bootstrap."
    )
sys.path.insert(0, str(PACKAGE_SOURCE))
importlib.invalidate_caches()
from vietmedbridge.artifacts import atomic_json, runtime_versions
from vietmedbridge import PIPELINE_API_VERSION
if PIPELINE_API_VERSION != EXPECTED_PIPELINE_API:
    raise RuntimeError("Notebook và package API không khớp; chọn commit data-v2 và restart session.")
_VMB_IMPORTED_COMMIT = CODE_COMMIT
if not lock or CODE_REVISION or upgrade:
    atomic_json(lock_path, {"repo_url": REPO_URL, "git_commit": CODE_COMMIT, "pipeline_api": PIPELINE_API_VERSION, "workflow_api": RETRIEVAL_WORKFLOW_API})
PIPELINE_CONFIG = json.loads((CHECKOUT / "configs/data_pipeline.json").read_text())
atomic_json(DATA_ROOT / "training_runtime.json", {
    "git_commit": CODE_COMMIT, "python": sys.version, "packages": runtime_versions()
})
print("Code commit:", CODE_COMMIT)
import vietmedbridge
print("Package source:", Path(vietmedbridge.__file__).resolve())
print("Dữ liệu/checkpoint:", DATA_ROOT)


## 2. Chuẩn bị nhãn — GPU chỉ khi còn thiếu draft train


In [ ]:
import json
from pathlib import Path
from vietmedbridge.training_data import prepare_training_data, install_ai_pilot_review
AI_REVIEW_UPLOAD = Path("/content/source_review_ai_pilot.json")
if AI_REVIEW_UPLOAD.is_file():
    print(json.dumps(install_ai_pilot_review(AI_REVIEW_UPLOAD, DATA_ROOT), ensure_ascii=False, indent=2))
PREPARATION = prepare_training_data(DATA_ROOT, CHECKOUT, max_new_samples=None)
print(json.dumps(PREPARATION, ensure_ascii=False, indent=2))
if PREPARATION["state"] == "WAITING_FOR_SOURCE_QUERY_REVIEW":
    from google.colab import files
    files.download(PREPARATION["review_path"])
    print("Duyệt file, lưu lại đúng review_path trên Drive, rồi Run all lại.")


## 3. Hybrid mining → duyệt hard negatives → QLoRA — GPU

Sau khi duyệt source queries, retriever chạy thật trên các query độc lập.
Chưa đủ positives/7 reviewed negatives mỗi query thì tải review.json chứa
source candidates/ranks. Team đánh POSITIVE/NEGATIVE/SKIP và reviewer;
category có thể để trống (unclassified). Không suy diễn missing = negative.
Lưu file về đúng review_path trên Drive rồi Run all lại. Vectors và scored
stages được reuse; dữ liệu training thay đổi có checkpoint namespace mới.

Nếu Sếp giao Codex duyệt, upload review_ai_pilot.json vào /content;
cell sau tự nhập vào đúng review_path, giữ backup và quyết định team.
Giữ checkpoint/run cũ. Khi còn chờ review, không cần giữ runtime GPU.


In [ ]:
from vietmedbridge.training_workflow import run_training_workflow
from vietmedbridge.negative_review import install_ai_candidate_review
NEGATIVE_REVIEW_UPLOAD = Path("/content/review_ai_pilot.json")
if NEGATIVE_REVIEW_UPLOAD.is_file():
    print(json.dumps(install_ai_candidate_review(NEGATIVE_REVIEW_UPLOAD, DATA_ROOT), ensure_ascii=False, indent=2))
if PREPARATION["state"] == "READY_FOR_HARD_NEGATIVE_REVIEW":
    TRAINING_STATUS = run_training_workflow(DATA_ROOT, CHECKOUT,
        work_dir=WORK_DIR, run_name="stage-a-qlora-v3-per-model-15b")
else:
    TRAINING_STATUS = {"state": "WAITING_FOR_SOURCE_QUERY_REVIEW", "fine_tuned": False}
print(json.dumps(TRAINING_STATUS, ensure_ascii=False, indent=2))
if TRAINING_STATUS.get("review_path"):
    from google.colab import files
    files.download(TRAINING_STATUS["review_path"])


Dừng ở bước review nghĩa là cần nhãn nguồn hoặc hard negatives; chưa có model fine-tune
hoặc F2 được chứng minh. Dev F2, ablation và held-out đều là local proxy,
chưa phải score BTC. Embedding fine-tune/sharded dense có API trong package,
hướng dẫn ở docs/FULL_PLAN.md. Sau khi chọn adapter hợp lệ, mở 04 và Run all;
không chỉnh ACTION hoặc xóa run pretrained. Nhãn/config đổi cần run mới.
